# Contratación pública: SECOP II, contratos electrónicos

Cada contrato que firman las entidades del Estado en la plataforma SECOP II: quién contrata, qué compra (código UNSPSC y objeto), con qué modalidad, por cuánto, durante cuánto tiempo y cuánto se ha pagado. Por defecto se descargan solo columnas útiles para modelar, **sin datos personales** (representantes legales, supervisores, cuentas bancarias). El cuaderno `analisis_secop.ipynb` de esta carpeta muestra un análisis completo de esta misma fuente.

|  |  |
|---|---|
| **Entidad** | Agencia Nacional de Contratación Pública – Colombia Compra Eficiente |
| **Conjunto** | SECOP II – Contratos Electrónicos · [`jbjy-vk9h`](https://www.datos.gov.co/d/jbjy-vk9h) |
| **Una fila es** | un contrato firmado |
| **Cobertura** | todo el país; `departamento` es el de la **entidad** que contrata |
| **Periodo** | junio de 2015 a hoy |
| **Tamaño total** | 5.671.200 contratos y 87 columnas (consultado el 31/10/2026) |
| **Actualización** | diaria |
| **Requisitos** | ninguno; para descargas de cientos de miles de filas se recomienda el token de datos.gov.co |
| **Tiempo de descarga** | 20 mil contratos: menos de 1 minuto · un año del país (≈ 1 millón): 15 a 30 minutos |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base



| Columna | Qué es | Tipo |
|---|---|---|
| `id_contrato`, `nombre_entidad`, `nit_entidad` | Identificador del contrato y entidad que contrata | texto |
| `departamento`, `ciudad` | Ubicación de la entidad | texto |
| `orden`, `sector`, `rama`, `entidad_centralizada` | Nacional o territorial, sector, rama del poder | categórica |
| `codigo_de_categoria_principal`, `segmento` | Código UNSPSC (p. ej. `V1.80111600`) y sus 2 primeros dígitos (segmento) | categórica |
| `objeto_del_contrato` | Descripción libre de lo que se contrata | texto libre |
| `tipo_de_contrato` | Prestación de servicios, Compraventa, Obra… | categórica |
| `modalidad_de_contratacion` | Contratación directa, Mínima cuantía, Licitación pública… | categórica |
| `fecha_de_firma`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato` | Fechas del contrato | fecha |
| `duracion_dias` | Días entre inicio y fin (calculada) | número |
| `es_pyme`, `es_grupo` | Si el proveedor es pyme o un grupo (consorcio, unión temporal) | binaria |
| `origen_de_los_recursos`, `destino_gasto`, `espostconflicto` | Origen y destino del presupuesto; si es del acuerdo de paz | categórica |
| `estado_contrato` | En ejecución, Modificado, Cerrado, terminado… | categórica |
| `valor_del_contrato`, `log_valor` | Valor en pesos y su logaritmo en base 10 (calculado) | número |
| `valor_pagado`, `dias_adicionados` | Lo pagado a la fecha y los días de prórroga | número |

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Regresión | `log_valor` (valor del contrato) | modalidad, tipo, segmento UNSPSC, entidad, duración, mes de firma |
| Clasificación | ¿El contrato es de contratación directa? | tipo, segmento, valor, entidad, sector |
| Clasificación | ¿El contrato tuvo prórroga (`dias_adicionados` > 0)? | lo conocido al firmar: valor, duración, modalidad, tipo |
| Texto (PLN) | `segmento` UNSPSC a partir de `objeto_del_contrato` | palabras del objeto (TF-IDF) |
| Anomalías | contratos con valores o plazos raros para su tipo | valor, duración, adiciones |

**Cuidados antes de modelar**

- **Fuga de información:** `valor_pagado`, `dias_adicionados` y `estado_contrato` se conocen **después** de firmar. No los use para predecir algo que se decide al firmar.
- El valor tiene una cola muy larga (de miles de pesos a billones): modele el logaritmo.
- Los contratos de una misma entidad se parecen mucho: valide por entidad (`GroupKFold`) o en el tiempo.
- `MAX_FILAS` toma los primeros contratos según la fecha de firma: no es una muestra al azar del periodo.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

El departamento y la modalidad se escriben con o sin tildes; la modalidad busca el texto dentro del nombre (`"directa"` trae *Contratación directa* y *Contratación Directa (con ofertas)*).

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
FECHA_INI    = "2025-01-01"   # fecha de firma desde (los datos empiezan en 2015)
FECHA_FIN    = "2025-12-31"   # fecha de firma hasta
DEPARTAMENTO = "Meta"         # departamento de la entidad, con o sin tildes. None = todo el país
MODALIDAD    = None           # texto dentro de la modalidad, p. ej. "directa", "licitacion", "minima". None = todas
VALOR_MIN    = 1              # valor mínimo del contrato en pesos (1 deja por fuera contratos en 0)
MAX_FILAS    = 20_000         # tope de filas. None = todas las que cumplan el filtro

COLUMNAS = [                  # sin datos personales; agregue o quite columnas de la tabla de variables
    "id_contrato", "nombre_entidad", "nit_entidad", "departamento", "ciudad", "orden", "sector", "rama",
    "entidad_centralizada", "codigo_de_categoria_principal", "objeto_del_contrato", "tipo_de_contrato",
    "modalidad_de_contratacion", "fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato",
    "es_pyme", "es_grupo", "origen_de_los_recursos", "destino_gasto", "espostconflicto", "estado_contrato",
    "valor_del_contrato", "valor_pagado", "dias_adicionados",
]

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. ¿Cuántos contratos cumplen el filtro?

In [ ]:
SECOP = "jbjy-vk9h"
filtros = [f"fecha_de_firma between '{FECHA_INI}T00:00:00' and '{FECHA_FIN}T23:59:59'",
           f"valor_del_contrato >= {VALOR_MIN}"]
if DEPARTAMENTO:
    filtros.append(f"departamento IN {lista_soql(valores_de(SECOP, 'departamento', DEPARTAMENTO))}")
if MODALIDAD:
    mods = valores_de(SECOP, "modalidad_de_contratacion", MODALIDAD,
                      where=f"fecha_de_firma >= '{FECHA_INI}T00:00:00'", contiene=True)
    print("Modalidades:", mods)
    filtros.append(f"modalidad_de_contratacion IN {lista_soql(mods)}")
WHERE = " AND ".join(filtros)

n = contar(SECOP, WHERE)
print("Filtro:", WHERE)
print(f"Contratos que cumplen el filtro: {entero(n)}")
if MAX_FILAS and n > MAX_FILAS:
    print(f"Se descargarán solo los primeros {entero(MAX_FILAS)} por fecha de firma (MAX_FILAS).")

## 5. Descarga

In [ ]:
secop = socrata(SECOP, select=", ".join(COLUMNAS), where=WHERE, order="fecha_de_firma, id_contrato",
                max_rows=MAX_FILAS)
print(f"Descargados {entero(len(secop))} contratos")

## 6. Limpieza mínima

Fechas y valores a su tipo, duración en días, logaritmo del valor y segmento UNSPSC (los dos primeros dígitos del código).

In [ ]:
for c in ["fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"]:
    if c in secop:
        secop[c] = pd.to_datetime(secop[c], errors="coerce")
secop = a_numero(secop, ["valor_del_contrato", "valor_pagado", "dias_adicionados", "nit_entidad"])
if {"fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"} <= set(secop.columns):
    secop["duracion_dias"] = (secop["fecha_de_fin_del_contrato"] - secop["fecha_de_inicio_del_contrato"]).dt.days
secop["log_valor"] = np.log10(secop["valor_del_contrato"])
if "codigo_de_categoria_principal" in secop:
    secop["segmento"] = secop["codigo_de_categoria_principal"].str.extract(r"(\d{2})", expand=False)
secop = secop.drop_duplicates("id_contrato")

## 7. Revisión de la base

In [ ]:
ficha_rapida(secop)
secop.head()

*(Se completa al ejecutar.)*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.hist(secop["log_valor"].dropna(), bins=60, color=AZUL, edgecolor="white", linewidth=0.4)
a1.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"$10^{x:.0f}$"))
a1.set_title("Valor del contrato (pesos, escala logarítmica)")
mod = secop["modalidad_de_contratacion"].value_counts(normalize=True).head(6).sort_values() * 100
a2.barh(mod.index, mod.values, color=NARANJA); a2.set_title("% de contratos por modalidad")
plt.tight_layout(); plt.show()
print("Mediana del valor:", entero(secop["valor_del_contrato"].median()), "pesos")

*(Se completa al ejecutar.)*

## 8. Guardar

In [ ]:
ruta = guardar(secop, etiqueta("secop_contratos", DEPARTAMENTO or "pais", FECHA_INI[:4], MODALIDAD))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/secop_contratos_meta_2025.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.